# Whisper — fix and sweep

**Established so far.** `large-v3-turbo` returned `en` on 75 of 99 clips and
translated rather than transcribed; delegating language ID to `large-v3` fixes
that. Both variants then produce roughly **half** the reference words, with ~50%
of the error being deletions. Beam size and `condition_on_previous_text` were
A/B tested across all four combinations and change nothing. At least part of it
is repetition collapse — one clip transcribes 47 s correctly then repeats a
single five-word phrase for the remaining 550 s.

**Sarvam reaches ratio 0.96 on this same audio**, so the reference word counts
are right and the audio is transcribable. The difference is that Sarvam runs
**per-segment**: handed one diarized turn at a time, it must return something
for each, while long-form Whisper decides for itself what to skip.

This notebook tests that directly, then sweeps if it holds. Turns come from the
**fusion**, which is also what the Sarvam sweep now uses — so the two systems
differ only in the recogniser.

In [8]:
# --- sync + config ----------------------------------------------------------
import subprocess, sys
from pathlib import Path

CODE = Path("/kaggle/working/sarvam-assignment")
if (CODE/".git").exists():
    subprocess.run(["git","-C",str(CODE),"fetch","-q","origin","main"], check=True)
    subprocess.run(["git","-C",str(CODE),"reset","-q","--hard","origin/main"], check=True)
else:
    subprocess.run(["git","clone","-q",
                    "https://github.com/ParvGoyal08/MultilingualASR.git", str(CODE)], check=True)
print("code @", subprocess.run(["git","-C",str(CODE),"log","-1","--format=%h %s"],
                               capture_output=True, text=True).stdout.strip())
for m in [k for k in list(sys.modules) if k=="sarvam_diar" or k.startswith("sarvam_diar.")]:
    del sys.modules[m]
sys.path.insert(0, str(CODE))

from sarvam_diar.config import Config, StageFlags
from sarvam_diar import asr, data, diarization, reference, refinement, text_metrics as tm
import torch, dataclasses, collections

ROOT = "/kaggle/working/sarvam_diarization"
cfg = Config.create(root=ROOT, work_dir=f"{ROOT}/tmp")
CLIPS = {c.clip_id: c for c in data.parse_ground_truth(data.load_segments_csv(cfg))}
INPUTS, _ = data.split_reference(list(CLIPS.values()), None, cfg=cfg)
REFS = {cid: reference.build_reference(CLIPS[cid]) for cid in INPUTS}
READY = [dataclasses.replace(ci, wav_path=str(cfg.wav_path(cid)))
         for cid, ci in INPUTS.items() if cfg.wav_path(cid).exists()]

MODEL = "large-v3-turbo"
SYSTEM = f"whisper-{MODEL}"
FUSION_MODELS = ["community-1", "reverb-v2", "diarizen-large"]
DIAR = "fusion"

# Report every device. Both Whisper models default to cuda:0, so on a 2xT4
# session ~5 GB of weights plus both activation peaks land on one card and it
# runs out of memory while the other sits idle. asr._lid_device() now puts
# language ID on the second card when there is one.
if torch.cuda.is_available():
    for _d in range(torch.cuda.device_count()):
        _free, _tot = torch.cuda.mem_get_info(_d)
        print(f"cuda:{_d} {torch.cuda.get_device_name(_d)}  "
              f"{_free/2**30:.1f} GB free of {_tot/2**30:.1f} GB")
    print("language ID will use cuda:", asr._lid_device())
else:
    print("no GPU")
print("clips with audio:", len(READY))

# the fusion must exist on disk before per-segment ASR can cut on it
refinement.materialise(cfg, REFS, FUSION_MODELS, threshold=0.5)
print("fusion clips:", sum(1 for c in REFS if diarization.is_done(cfg, DIAR, c)))


code @ 2c4cd7d main_kaggle_2: test the per-segment hypothesis, then sweep
12:05:43 | INFO    | sarvam_diar | segments CSV already cached (/kaggle/working/sarvam_diarization/data/youtube_segments.csv)
12:05:44 | INFO    | sarvam_diar | parsed 100 clips, 9940 segments, 2 dropped as malformed, 0 unparsable entries
GPU: Tesla T4
clips with audio: 99
12:05:45 | INFO    | sarvam_diar | materialised 'fusion' for 99 clips from ['community-1', 'reverb-v2', 'diarizen-large']
fusion clips: 99


## 1 — long-form vs per-segment

In [9]:
# --- the decisive test: does per-segment fix the deletions? -----------------
# Long-form Whisper decides for itself what to transcribe and drops roughly half
# the words. Sarvam reaches ratio 0.96 on the same audio running PER SEGMENT,
# which removes that choice: it is handed one turn at a time and must return
# something for each. If that is the difference, Whisper per-segment should also
# land near 1.0 -- and it is the strategy that makes the two systems comparable.
import time, gc

def _free_gpu():
    """Drop cached blocks between configurations.

    Three configurations run back to back in one kernel, and CTranslate2 holds
    its workspace until the model object is released. Without this the peak is
    the sum of the configurations rather than the largest of them.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


probe = sorted(READY, key=lambda c: c.duration)[:3]
print(f"{'strategy':<34}{'ref w':>7}{'hyp w':>7}{'ratio':>7}{'WER':>9}{'del%':>7}{'sec':>6}")

def score(pairs_by_clip):
    rw = hw = d = s_ = i_ = h = 0
    for cid, hyp in pairs_by_clip.items():
        ref = REFS[cid]
        r = [t for u in ref.utterances for t in reference.tokenize(u.text_norm)]
        c = tm.wer_counts(r, hyp)
        rw += len(r); hw += len(hyp)
        d += c.deletions; s_ += c.substitutions; i_ += c.insertions; h += c.hits
    n = h + s_ + d
    return rw, hw, hw/max(rw,1), (s_+d+i_)/max(n,1), d/max(n,1)

# long-form, current settings
t0 = time.time(); out = {}
for c in probe:
    words, _ = asr.transcribe_whisper(cfg, Path(c.wav_path), word_timestamps=False,
                                      beam_size=5, lid_model=asr.LID_MODEL)
    out[c.clip_id] = [t for w in words
                      for t in reference.normalize_text(w.text, strip_gloss=False).split()]
print(f"{'long-form (current)':<34}" + "{:>7}{:>7}{:>7.2f}{:>9.4f}{:>7.1%}".format(*score(out))
      + f"{time.time()-t0:>6.0f}")

# long-form with the discard threshold off
t0 = time.time(); out = {}
for c in probe:
    words, _ = asr.transcribe_whisper(cfg, Path(c.wav_path), word_timestamps=False,
                                      beam_size=5, lid_model=asr.LID_MODEL,
                                      no_speech_threshold=None)
    out[c.clip_id] = [t for w in words
                      for t in reference.normalize_text(w.text, strip_gloss=False).split()]
_free_gpu()
print(f"{'long-form, no_speech off':<34}" + "{:>7}{:>7}{:>7.2f}{:>9.4f}{:>7.1%}".format(*score(out))
      + f"{time.time()-t0:>6.0f}")

# per-segment over the fusion, exactly how Sarvam is run
t0 = time.time(); out = {}
for c in probe:
    turns = asr.merge_same_speaker(diarization.load_hypothesis(cfg, DIAR, c.clip_id), 1.0)
    segs, _m = asr.transcribe_segments(cfg, SYSTEM, Path(c.wav_path), turns)
    out[c.clip_id] = [t for sg in segs
                      for t in reference.normalize_text(sg["text"], strip_gloss=False).split()]
_free_gpu()
print(f"{'PER-SEGMENT on fusion':<34}" + "{:>7}{:>7}{:>7.2f}{:>9.4f}{:>7.1%}".format(*score(out))
      + f"{time.time()-t0:>6.0f}")

print("\nratio near 1.0 = producing about as many words as were spoken.")
print("Sarvam per-segment on this corpus: 0.96.")


strategy                            ref w  hyp w  ratio      WER   del%   sec
12:05:45 | INFO    | sarvam_diar | loading faster-whisper large-v3 on cuda (float16) -- the first call also downloads the weights


RuntimeError: CUDA failed with error out of memory

## 2 — sweep (set `RUN = True` only if the test justifies it)

In [ ]:
# --- full per-segment sweep, ONLY if the test above showed ratio near 1.0 ----
# Same strategy and the same turns as Sarvam, so the two are comparable on the
# recogniser alone rather than on how the audio was cut.
RUN = False          # flip to True once the test above justifies it

if RUN:
    m = asr.run_segmented(cfg, READY, diar_model=DIAR, systems=[SYSTEM],
                          flags=StageFlags(), merge_gap=1.0)
    if len(m):
        display(m[["clip_id","n_segments","n_words","elapsed_sec","rtf",
                   "detected_language"]].head(10))
else:
    print("RUN is False -- inspect the test output first")


## 3 — score

In [ ]:
# --- score whatever is on disk ---------------------------------------------
import json as _json
_norm = lambda t: reference.normalize_text(t, strip_gloss=False)

def _pairs(system, cid):
    p = _json.loads(asr.asr_path(cfg, system, cid).read_text())
    if p.get("strategy") == "segment":
        return [(t, s["speaker"]) for s in sorted(p["segments"], key=lambda s: s["start"])
                for t in _norm(s["text"]).split()]
    turns = diarization.load_hypothesis(cfg, DIAR, cid)
    return [(t, spk) for w, spk in asr.assign_words(asr.load_words(cfg, system, cid), turns)
            for t in _norm(w).split()]

_found = sorted(d.name for d in (cfg.root/"asr").iterdir()) if (cfg.root/"asr").exists() else []
print(f"{'system':<34}{'clips':>6}{'ratio':>7}{'WER':>9}{'cpWER':>9}{'WDER':>8}")
for _s in _found:
    rows = []
    for cid, ref in REFS.items():
        if not asr.is_done(cfg, _s, cid):
            continue
        if "@" not in _s and not diarization.is_done(cfg, DIAR, cid):
            continue
        pr = _pairs(_s, cid)
        r = tm.score_transcript(
            {k: v.split() for k, v in reference.speaker_texts(ref).items()},
            asr.speaker_texts_from_words(pr), reference.word_stream(ref), pr)
        r["n_hyp"] = len(pr); rows.append(r)
    if not rows:
        continue
    g = tm.summarise(rows)
    ratio = sum(r["n_hyp"] for r in rows)/max(g["n_ref_words"],1)
    print(f"{_s:<34}{g['n_clips']:>6}{ratio:>7.2f}{g['wer']:>9.4f}"
          f"{g['cpwer']:>9.4f}{g['wder']:>8.4f}")
